# Lakebase migrations and app grants

Idempotent setup for the app's Postgres database, run by the `pdu-lakebase-setup` job after `bundle deploy`:

1. Applies `src/sql/lakebase/NNN_*.sql` files in order, each once, tracked in `pdu_app.schema_migrations`.
2. Grants the **app's service principal** (its Postgres role is its client ID) read/write on the operator-state tables and read on the synced `pdu_latest_status_synced` table.
3. Verifies the grants with `has_table_privilege`.

Runs as the deploying identity (the Lakebase project owner).

In [0]:
%pip install -q "psycopg[binary]>=3.1" "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("lakebase_endpoint", "projects/pdu-monitoring-dev/branches/production/endpoints/primary")
dbutils.widgets.text("database", "pdu_monitoring")
dbutils.widgets.text("app_name", "")
dbutils.widgets.text("migrations_dir", "")
dbutils.widgets.text("synced_schema", "pdu_monitoring")
dbutils.widgets.text("synced_table", "pdu_latest_status_synced")

ENDPOINT = dbutils.widgets.get("lakebase_endpoint")
DATABASE = dbutils.widgets.get("database")
APP_NAME = dbutils.widgets.get("app_name")
MIGRATIONS_DIR = dbutils.widgets.get("migrations_dir")
SYNCED_SCHEMA = dbutils.widgets.get("synced_schema")
SYNCED_TABLE = dbutils.widgets.get("synced_table")

import os, re
if not MIGRATIONS_DIR:
    nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    MIGRATIONS_DIR = os.path.normpath(os.path.join("/Workspace" + os.path.dirname(nb), "..", "sql", "lakebase"))
print(ENDPOINT, DATABASE, APP_NAME, MIGRATIONS_DIR)

In [0]:
import psycopg
from psycopg import sql
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.postgres.get_endpoint(name=ENDPOINT).status.hosts.host
me = w.current_user.me().user_name


def connect():
    token = w.postgres.generate_database_credential(endpoint=ENDPOINT).token
    return psycopg.connect(host=host, dbname=DATABASE, user=me, password=token, sslmode="require", autocommit=False)


with connect() as conn:
    print(conn.execute("select current_user, current_database(), version()").fetchone())

In [0]:
files = sorted(f for f in os.listdir(MIGRATIONS_DIR) if re.fullmatch(r"\d{3}_[a-z0-9_]+\.sql", f))
print("found:", files)

with connect() as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS pdu_app")
    conn.execute("""CREATE TABLE IF NOT EXISTS pdu_app.schema_migrations (
                      version TEXT PRIMARY KEY, applied_at TIMESTAMPTZ NOT NULL DEFAULT now(), applied_by TEXT NOT NULL)""")
    conn.commit()
    applied = {r[0] for r in conn.execute("SELECT version FROM pdu_app.schema_migrations")}
    for f in files:
        if f in applied:
            print("skip   ", f)
            continue
        with open(os.path.join(MIGRATIONS_DIR, f)) as fh:
            ddl = fh.read()
        # ClientCursor uses the simple query protocol, so a file may contain several statements.
        with psycopg.ClientCursor(conn) as cur:
            cur.execute(ddl)
            cur.execute("INSERT INTO pdu_app.schema_migrations (version, applied_by) VALUES (%s, %s)", (f, me))
        conn.commit()  # one transaction per migration file
        print("applied", f)

In [0]:
if not APP_NAME:
    dbutils.notebook.exit("No app_name given; migrations applied, grants skipped.")

app_role = w.apps.get(APP_NAME).service_principal_client_id
print("app service principal role:", app_role)
R = sql.Identifier(app_role)

with connect() as conn:
    if not conn.execute("SELECT 1 FROM pg_roles WHERE rolname = %s", (app_role,)).fetchone():
        raise RuntimeError(f"Postgres role {app_role} not found - deploy the app (with its Lakebase resource) first.")

    # Operator-state tables: read/write (no DELETE - acks and notes are an audit trail).
    conn.execute(sql.SQL("GRANT USAGE ON SCHEMA pdu_app TO {}").format(R))
    conn.execute(sql.SQL("GRANT SELECT, INSERT, UPDATE ON pdu_app.incident_ack, pdu_app.unit_notes TO {}").format(R))
    conn.execute(sql.SQL("GRANT SELECT ON pdu_app.schema_migrations TO {}").format(R))
    conn.execute(sql.SQL("ALTER DEFAULT PRIVILEGES IN SCHEMA pdu_app GRANT SELECT, INSERT, UPDATE ON TABLES TO {}").format(R))

    # Synced table: read-only. The sync pipeline owns (and may re-create) the table, so prefer the
    # database's synced-table reader role, which keeps working across re-snapshots.
    reader = conn.execute("SELECT 'databricks_reader_' || oid FROM pg_database WHERE datname = current_database()").fetchone()[0]
    conn.execute(sql.SQL("GRANT USAGE ON SCHEMA {} TO {}").format(sql.Identifier(SYNCED_SCHEMA), R))
    try:
        with conn.transaction():
            conn.execute(sql.SQL("GRANT {} TO {}").format(sql.Identifier(reader), R))
        print("granted reader role", reader)
    except psycopg.Error as e:
        print(f"could not grant {reader} ({e.diag.message_primary}); granting SELECT on the table directly")
        conn.execute(sql.SQL("GRANT SELECT ON {}.{} TO {}").format(sql.Identifier(SYNCED_SCHEMA), sql.Identifier(SYNCED_TABLE), R))
    conn.commit()

In [0]:
checks = [
    (f"{SYNCED_SCHEMA}.{SYNCED_TABLE}", "SELECT", True),
    ("pdu_app.incident_ack", "SELECT", True), ("pdu_app.incident_ack", "INSERT", True), ("pdu_app.incident_ack", "DELETE", False),
    ("pdu_app.unit_notes", "SELECT", True), ("pdu_app.unit_notes", "INSERT", True),
]
with connect() as conn:
    results = []
    for table, priv, expected in checks:
        ok = conn.execute("SELECT has_table_privilege(%s, %s, %s)", (app_role, table, priv)).fetchone()[0]
        results.append((table, priv, ok, expected))
        print(f"{'PASS' if ok == expected else 'FAIL'}  {app_role} {priv:<6} {table}: {ok}")
if any(ok != exp for _, _, ok, exp in results):
    raise AssertionError("App service principal privileges are not as expected")